In [3]:
import sqlite3 #загрузка библиотек
import pandas as pd
import numpy as np

In [17]:
db_path = "C:/Users/Home/Downloads/W3 Data (1)/W3 Data/w3_database.sqlite" #подключение к SQLite БД 
conn = sqlite3.connect(db_path)

In [19]:
coins = pd.read_sql("SELECT * FROM coins", conn) #загрузка табл coins из БД в DataFrame 
print(coins.head())
print(coins.shape)

   id name symbol  cmc_id explorers socials
0   1  BTC    BTC       0        []      []
1   2  ETH    ETH       0        []      []
2   4  BNB    BNB       0        []      []
3   5  XRP    XRP       0        []      []
4   7  SOL    SOL       0        []      []
(3375, 6)


Задание 1. Точное сопоставление

In [26]:
import json

with open("C:/Users/Home/Downloads/W3 Data (1)/W3 Data/cmc_latest.json", "r", encoding="utf-8") as f:
    cmc_raw = json.load(f)

cmc_df = pd.DataFrame([ #для каждой монеты извлечен id, символ, ранг и цену в USD
    {
        "cmc_id":   c["id"],
        "symbol":   c["symbol"].upper(),
        "cmc_rank": c.get("cmc_rank") or 99999,
        "price":    (c.get("quote") or {}).get("USD", {}).get("price")
    }
    for c in cmc_raw  
])

cmc_df.to_sql("cmc", conn, if_exists="replace", index=False) #Сохр DataFrame в таблицу cmc в БД
print("cmc загружена:", len(cmc_df), "монет")

cmc загружена: 8740 монет


In [28]:
cursor = conn.cursor()

for col in ["ALTER TABLE coins ADD COLUMN cmc_price REAL", #добавляем колонки cmc_price и match_method 
            "ALTER TABLE coins ADD COLUMN match_method TEXT"]:
    try: cursor.execute(col)
    except: pass #пропуск если колонка уже есть

    #сопоставление монет по символу: запись с наименьшим рангом из CMC
cursor.execute(""" 
    UPDATE coins
    SET
        cmc_id       = (SELECT cmc_id FROM cmc WHERE cmc.symbol = coins.symbol ORDER BY cmc_rank ASC LIMIT 1),
        cmc_price    = (SELECT price  FROM cmc WHERE cmc.symbol = coins.symbol ORDER BY cmc_rank ASC LIMIT 1),
        match_method = 'symbol+rank'
    WHERE coins.cmc_id = 0
      AND EXISTS (SELECT 1 FROM cmc WHERE cmc.symbol = coins.symbol)
""")
conn.commit()
#итоги: сколько монет сопоставлено и сколько осталось без match
result = pd.read_sql("""
    SELECT
        SUM(CASE WHEN cmc_id != 0 THEN 1 ELSE 0 END) as сопоставлено,
        SUM(CASE WHEN cmc_id  = 0 THEN 1 ELSE 0 END) as не_найдено
    FROM coins
""", conn)
print(result)

   сопоставлено  не_найдено
0          2883         492


Задание 2. Нечёткое сопоставление

In [32]:
!pip install rapidfuzz
from rapidfuzz import fuzz

#несопоставленные монеты из coins
unmatched = pd.read_sql("SELECT id, symbol, cmc_price FROM coins WHERE cmc_id = 0", conn)

#все монеты CMC с ценой
cmc_list = cmc_df[["cmc_id", "symbol", "price", "cmc_rank"]].dropna().to_dict("records")

#макс порог отличия цены  20%
PRICE_THRESHOLD = 0.20

def price_close(p1, p2):
    #True если цены отличаются не более чем на 20%
    if not p1 or not p2 or p2 == 0:
        return False
    return abs(p1 - p2) / p2 <= PRICE_THRESHOLD

In [35]:
cursor = conn.cursor()
fuzzy_matched = 0

unmatched = pd.read_sql("SELECT id, symbol FROM coins WHERE cmc_id = 0", conn)

for _, row in unmatched.iterrows():
    best_score = 0
    best = None

    for c in cmc_list:
        #только схожесть тикера больше или равно 80
        score = fuzz.ratio(row["symbol"].upper(), c["symbol"].upper())
        if score >= 80 and score > best_score:
            best_score = score
            best = c

    if best:
        cursor.execute("""
            UPDATE coins
            SET cmc_id = ?, cmc_price = ?, match_method = ?
            WHERE id = ?
        """, (best["cmc_id"], best["price"], f"fuzzy:{best_score}", row["id"]))
        fuzzy_matched += 1

conn.commit()
print(f"Дополнительно сопоставлено: {fuzzy_matched} монет")

Дополнительно сопоставлено: 342 монет


In [36]:
result = pd.read_sql("""
    SELECT match_method, COUNT(*) as кол_во
    FROM coins
    GROUP BY match_method
""", conn)
print(result)

              match_method  кол_во
0                     None     150
1               fuzzy:80.0      91
2  fuzzy:83.33333333333334       4
3  fuzzy:85.71428571428572     175
4  fuzzy:88.88888888888889      58
5   fuzzy:90.9090909090909      14
6              symbol+rank    2883


Задание 3. Наивное сопоставление и добавление новых монет

In [37]:
#назначаем cmc_id оставшимся монетам по symbol из cmc_map

unmatched = pd.read_sql("SELECT id, symbol FROM coins WHERE cmc_id = 0", conn)

#использованные cmc_id
used_ids = set(pd.read_sql("SELECT cmc_id FROM coins WHERE cmc_id != 0", conn)["cmc_id"])

cursor = conn.cursor()
assigned = 0

for _, row in unmatched.iterrows():
    #первый подходящий по symbol 
    candidates = cmc_df[
        (cmc_df["symbol"] == row["symbol"].upper()) &
        (~cmc_df["cmc_id"].isin(used_ids))
    ]

    if not candidates.empty:
        best = candidates.iloc[0]
        cursor.execute("""
            UPDATE coins SET cmc_id = ?, match_method = ?
            WHERE id = ?
        """, (int(best["cmc_id"]), "cmc_map:symbol", row["id"]))
        used_ids.add(best["cmc_id"])
        assigned += 1

conn.commit()
print(f"Назначено через cmc_map: {assigned} монет")

Назначено через cmc_map: 0 монет


In [40]:
#сопоставленные cmc_id в coins
existing_ids = set(pd.read_sql("SELECT cmc_id FROM coins WHERE cmc_id != 0", conn)["cmc_id"])

#монеты из CMC которых нет в coins 
new_coins = cmc_df[~cmc_df["cmc_id"].isin(existing_ids)]
print(f"Новых монет для добавления: {len(new_coins)}")

#добавляем через INSERT 
cursor = conn.cursor()
added = 0

for _, c in new_coins.iterrows():
    cursor.execute("""
        INSERT INTO coins (name, symbol, cmc_id, match_method)
        VALUES (?, ?, ?, ?)
    """, (c["symbol"], c["symbol"], int(c["cmc_id"]), "cmc_map:new"))
    added += 1

conn.commit()
print(f"Добавлено новых монет: {added}")

Новых монет для добавления: 5894
Добавлено новых монет: 5894


In [41]:
result = pd.read_sql("SELECT COUNT(*) as всего FROM coins", conn)
print(result)

   всего
0   9269


Задание 4.Загрузка name и explorers

In [60]:
import sqlite3, json, requests, time

API_KEY  = "771047f82f464c4cb89014739802f980"
BASE_URL = "https://pro-api.coinmarketcap.com"
DB_PATH  = r"C:\Users\Home\Downloads\W3 Data (1)\W3 Data\w3_database.sqlite"

conn = sqlite3.connect(DB_PATH)
all_ids = [row[0] for row in conn.execute("SELECT cmc_id FROM coins WHERE cmc_id != 0").fetchall()]
print(f"Всего монет с cmc_id: {len(all_ids)}, запускаю...")

updated = 0
errors = 0
for i in range(0, len(all_ids), 100):
    chunk = all_ids[i:i+100]
    try:
        resp = requests.get(
            f"{BASE_URL}/v2/cryptocurrency/info",
            params={"id": ",".join(map(str, chunk)), "aux": "urls"},
            headers={"X-CMC_PRO_API_KEY": API_KEY}
        )
        data = resp.json().get("data", {})
        for cmc_id in chunk:
            info = data.get(str(cmc_id), {})
            name = info.get("name")
            explorers = info.get("urls", {}).get("explorer", [])
            if name:
                conn.execute(
                    "UPDATE coins SET name=?, explorers=? WHERE cmc_id=?",
                    (name, json.dumps(explorers), cmc_id)
                )
                updated += 1
        conn.commit()
        time.sleep(0.5)
    except Exception as e:
        errors += 1

conn.close()
print(f"Готово! Обновлено: {updated} монет, ошибок: {errors}")

Всего монет с cmc_id: 9119, запускаю...
Готово! Обновлено: 8719 монет, ошибок: 0


Задание 5. Обязательные socials

In [ ]:
import sqlite3, json, requests, time

API_KEY = "771047f82f464c4cb89014739802f980"
DB_PATH = r"C:\Users\Home\Downloads\W3 Data (1)\W3 Data\w3_database.sqlite"

FIELDS = {
    "website": "website",
    "technical_doc": "whitepaper",
    "source_code": "github",
    "twitter": "twitter",
}

conn = sqlite3.connect(DB_PATH)
all_ids = [r[0] for r in conn.execute("SELECT cmc_id FROM coins WHERE cmc_id != 0").fetchall()]
print(f"Всего монет: {len(all_ids)}, запускаю...")

updated = errors = 0
for i in range(0, len(all_ids), 100):
    chunk = all_ids[i:i+100]
    try:
        data = requests.get(
            "https://pro-api.coinmarketcap.com/v2/cryptocurrency/info",
            params={"id": ",".join(map(str, chunk)), "aux": "urls"},
            headers={"X-CMC_PRO_API_KEY": API_KEY}
        ).json().get("data", {})
        for cmc_id in chunk:
            urls = data.get(str(cmc_id), {}).get("urls", {})
            socials = [{"type": t, "link": l} for f, t in FIELDS.items() for l in urls.get(f, []) if l]
            conn.execute("UPDATE coins SET socials=? WHERE cmc_id=?", (json.dumps(socials), cmc_id))
            updated += 1
        conn.commit()
        time.sleep(0.5)
    except Exception as e:
        errors += 1

conn.close()
print(f" Обновлено: {updated}, ошибок: {errors}")

Задание 7. 

In [68]:
import logging

LOG_PATH = r"C:\Users\Home\Downloads\W3 Data (1)\W3 Data\w3_log.log"
#Получаем логгер и сбрасываем старые обработчики 
logger = logging.getLogger("W3")
logger.handlers.clear()
logger.setLevel(logging.DEBUG)

fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
#файловый обработчик 
fh = logging.FileHandler(LOG_PATH, encoding="utf-8")
fh.setLevel(logging.DEBUG)
fh.setFormatter(fmt)
#консольный обработчик
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
ch.setFormatter(fmt)

logger.addHandler(fh)
logger.addHandler(ch)

logger.debug("Сравнение цен: BTC coins=67543.2, CMC=67550.1, разница=0.01%")
logger.info("Задание 1 завершено: 850 из 1000 монет сопоставлены")
logger.warning("Монета XYZ: найдено 3 кандидата в CMC, выбран по наименьшей разнице цены")
logger.error("Ошибка API при загрузке info для ID 12345: 429 Too Many Requests")

2026-05-19 15:40:37,080 [INFO] Задание 1 завершено: 850 из 1000 монет сопоставлены
2026-05-19 15:40:37,081 [WARNING] Монета XYZ: найдено 3 кандидата в CMC, выбран по наименьшей разнице цены
2026-05-19 15:40:37,083 [ERROR] Ошибка API при загрузке info для ID 12345: 429 Too Many Requests
